# Train — 鋼鐵缺陷分割訓練
**Detection of Manufacturing Defects in Steel Using Deep Learning**

流程：資料下載 → 前處理 / 擴增 → 損失函數 → 改良版模型CSAD Unet → 訓練 → 儲存


In [1]:
!pip install -q albumentations

from google.colab import drive, files
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, math, random, glob, zipfile
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers as L
from tensorflow.keras.utils import register_keras_serializable
import albumentations as A

# ── 路徑 ──────────────────────────────────────────────────────
DATA_DIR   = "/content/data"
TRAIN_CSV  = os.path.join(DATA_DIR, "train.csv")
TRAIN_DIR  = os.path.join(DATA_DIR, "train_images")
MODEL_DIR  = "/content/drive/MyDrive/實驗結果"

# ── 影像尺寸 ──────────────────────────────────────────────────
IMG_H, IMG_W = 128, 800
N_CLASSES    = 4

# ── 訓練超參數 ────────────────────────────────────────────────
EPOCHS = 30
BATCH  = 8
LR     = 1e-5

# ── 模型設定 ──────────────────────────────────────────────────
BACKBONE      = "efficientnetb4"   # 'resnet34' | 'inceptionv3' | 'efficientnetb4'
ARCH          = "unet"              # 'unet' | 'fpn'
NOISE_SETTING = "none"              # 'none' | 'low' | 'high'

# ── 各類別 Oversampling 倍數（補正 class imbalance）─────────────
UPSAMPLE_RATIO = {1: 6, 2: 12, 3: 3, 4: 6}

# ── 隨機種子 ──────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("TF:", tf.__version__)
print("Numpy:", np.__version__)
print("cv2:", cv2.__version__)
print("albumentations:", A.__version__)
print(f"ARCH={ARCH}  BACKBONE={BACKBONE}  NOISE={NOISE_SETTING}")


TF: 2.20.0
Numpy: 2.1.3
cv2: 5.0.0
albumentations: 2.0.8
ARCH=unet  BACKBONE=efficientnetb4  NOISE=none


In [ ]:
# 上傳 kaggle.json 金鑰後自動下載資料集
files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle competitions download -c severstal-steel-defect-detection -p {DATA_DIR}

for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
    with zipfile.ZipFile(z) as zz:
        zz.extractall(DATA_DIR)
print("✅ 解壓完成:", os.listdir(DATA_DIR))


Saving kaggle.json to kaggle.json
100% 1.57G/1.57G [01:17<00:00, 21.8MB/s]

✅ 解壓完成: ['test_images', 'sample_submission.csv', 'severstal-steel-defect-detection.zip', 'train.csv', 'train_images']


In [ ]:
def read_train_csv(path):
    """支援兩欄 / 三欄格式，統一輸出 [ImageId_ClassId, EncodedPixels]。"""
    if not os.path.exists(path):
        return pd.DataFrame(columns=["ImageId_ClassId", "EncodedPixels"])
    raw  = pd.read_csv(path)
    cols = set(map(str, raw.columns))
    if {"ImageId_ClassId", "EncodedPixels"}.issubset(cols):
        return raw[["ImageId_ClassId", "EncodedPixels"]].copy()
    if {"ImageId", "ClassId", "EncodedPixels"}.issubset(cols):
        df = raw[["ImageId", "ClassId", "EncodedPixels"]].copy()
        df["ImageId_ClassId"] = df["ImageId"].astype(str) + "_" + df["ClassId"].astype(str)
        return df[["ImageId_ClassId", "EncodedPixels"]]
    raise ValueError(f"Unexpected CSV columns: {list(raw.columns)}")

def rle_decode(mask_rle, shape):
    """RLE 字串 → (H,W) uint8 mask（Fortran order）。"""
    if not isinstance(mask_rle, str) or not mask_rle.strip():
        return np.zeros(shape, dtype=np.uint8)
    s = mask_rle.strip().split()
    starts  = np.asarray(s[0::2], dtype=int) - 1
    lengths = np.asarray(s[1::2], dtype=int)
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, starts + lengths):
        img[lo:hi] = 1
    return img.reshape(shape, order="F")

def rle_encode(img):
    pixels = img.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

# 讀取並統計
df = read_train_csv(TRAIN_CSV)
all_ids = sorted(os.path.basename(p)
                 for p in glob.glob(os.path.join(TRAIN_DIR, "*.jpg")))
print(f"總影像數：{len(all_ids)}")

has_mask = df["EncodedPixels"].notna() & (df["EncodedPixels"].astype(str).str.strip() != "")
df["ImageId"] = df["ImageId_ClassId"].astype(str).str.rsplit("_", n=1).str[0]
print(f"有缺陷：{df.loc[has_mask,'ImageId'].nunique()}  "
      f"無缺陷：{len(all_ids) - df.loc[has_mask,'ImageId'].nunique()}")


總影像數：12568
有缺陷：6666  無缺陷：5902


## data augmentation

In [ ]:
groups = df.groupby("ImageId")

rng = np.random.RandomState(RANDOM_SEED)
idx = rng.permutation(len(all_ids))
n   = len(all_ids)
a, b = int(0.6 * n), int(0.8 * n)

tr_raw = [all_ids[i] for i in idx[:a]]
va_ids = [all_ids[i] for i in idx[a:b]]
te_ids = [all_ids[i] for i in idx[b:]]

tr_ids = []
for img_id in tr_raw:
    repeat = 1
    if img_id in groups.groups:
        g       = groups.get_group(img_id)
        present = set()
        for _, row in g.iterrows():
            rle = row.get("EncodedPixels", "")
            if isinstance(rle, str) and rle.strip():
                present.add(int(str(row["ImageId_ClassId"]).rsplit("_",1)[1]))
        if present:
            repeat = max(UPSAMPLE_RATIO.get(c, 1) for c in present)
    tr_ids.extend([img_id] * repeat)

rng.shuffle(tr_ids)
print(f"Train raw={len(tr_raw)} → oversampled={len(tr_ids)}")
print(f"Val={len(va_ids)}  Test={len(te_ids)}")


Train raw=7540 → oversampled=19743
Val=2514  Test=2514


##計算照片瑕疵佔比像素

In [ ]:
from skimage.measure import label, regionprops

IMG_H_ORIG, IMG_W_ORIG = 256, 1600
TOTAL_PIX = IMG_H_ORIG * IMG_W_ORIG

stats = {c: {"img_count": 0, "pixel_counts": [],
             "region_counts": [], "max_axis": []} for c in range(1, 5)}

for img_id in all_ids:  # ← 用原始 all_ids，不是 tr_ids
    g = groups.get_group(img_id) if img_id in groups.groups else None
    if g is None:
        continue
    img0 = cv2.imread(os.path.join(TRAIN_DIR, img_id), cv2.IMREAD_GRAYSCALE)
    if img0 is None:
        continue
    H0, W0 = img0.shape
    mask4 = np.zeros((H0, W0, N_CLASSES), np.uint8)
    for _, row in g.iterrows():
        ch = int(str(row["ImageId_ClassId"]).rsplit("_", 1)[1]) - 1
        rle = row.get("EncodedPixels", "")
        if isinstance(rle, str) and rle.strip():
            mask4[..., ch] = rle_decode(rle, (H0, W0))

    for c in range(1, 5):
        m = mask4[..., c-1]
        if m.sum() == 0:
            continue
        stats[c]["img_count"] += 1
        stats[c]["pixel_counts"].append(m.sum())
        labeled = label(m)
        stats[c]["region_counts"].append(labeled.max())
        lengths = [max(r.bbox[2]-r.bbox[0], r.bbox[3]-r.bbox[1])
                   for r in regionprops(labeled)]
        stats[c]["max_axis"].append(max(lengths) if lengths else 0)

print("\n=== Defect Pixel-level Statistics ===")
for c in range(1, 5):
    s = stats[c]
    avg_pct    = np.mean(s["pixel_counts"]) / TOTAL_PIX * 100 if s["pixel_counts"] else 0
    avg_region = np.mean(s["region_counts"]) if s["region_counts"] else 0
    avg_axis   = np.mean(s["max_axis"]) if s["max_axis"] else 0
    print(f"Defect {c}: 含缺陷圖片數={s['img_count']}, "
          f"平均像素佔比={avg_pct:.3f}%, "
          f"平均連通區域數={avg_region:.1f}, "
          f"平均最長軸={avg_axis:.0f}px")


=== Defect Pixel-level Statistics ===
Defect 1: 含缺陷圖片數=897, 平均像素佔比=1.065%, 平均連通區域數=3.4, 平均最長軸=97px
Defect 2: 含缺陷圖片數=247, 平均像素佔比=0.825%, 平均連通區域數=1.3, 平均最長軸=185px
Defect 3: 含缺陷圖片數=5150, 平均像素佔比=6.225%, 平均連通區域數=2.8, 平均最長軸=282px
Defect 4: 含缺陷圖片數=801, 平均像素佔比=8.392%, 平均連通區域數=2.4, 平均最長軸=275px


* 用灰階讀入原始影像（Severstal 資料集，原始尺寸 1600×256）
*   建立一個 4 通道的遮罩 mask4（對應論文中的 D1~D4 四類缺陷），每個通道用 rle_decode 把 CSV 裡的 RLE 編碼字串還原成真正的二值遮罩，畫在正確的類別通道上
*   把影像和遮罩都縮放到模型輸入尺寸（IMG_W × IMG_H，也就是論文中的 128×800）
*   回傳 (img, mask) 這一組訓練樣本



In [ ]:
def load_image_and_mask(img_id, df_group):
    path = os.path.join(TRAIN_DIR, img_id)
    img0 = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img0 is None:
        img0 = np.zeros((256, 1600), np.uint8)
    H0, W0 = img0.shape
    mask4 = np.zeros((H0, W0, 4), np.uint8)  # ← 固定 4，不用 N_CLASSES
    if df_group is not None:
        for _, row in df_group.iterrows():
            ch  = int(str(row["ImageId_ClassId"]).rsplit("_",1)[1]) - 1
            rle = row.get("EncodedPixels", "")
            if isinstance(rle, str) and rle.strip():
                mask4[..., ch] = rle_decode(rle, (H0, W0))
    img  = cv2.resize(img0, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    mask = np.stack([cv2.resize(mask4[...,k], (IMG_W, IMG_H),
                                interpolation=cv2.INTER_NEAREST)
                     for k in range(4)], axis=-1)  # ← 固定 4
    return img, mask

def preprocess_image(img_u8):
    if img_u8.ndim == 2: img_u8 = img_u8[..., None]
    if img_u8.shape[-1] == 1: img_u8 = np.repeat(img_u8, 3, axis=-1)
    img = img_u8.astype("float32")
    if   BACKBONE == "resnet50":     return tf.keras.applications.resnet50.preprocess_input(img)
    elif BACKBONE == "inceptionv3":  return tf.keras.applications.inception_v3.preprocess_input(img)
    else:                            return tf.keras.applications.efficientnet.preprocess_input(img)


def apply_aug(img_u8, mask_u8, aug):
    if aug is None:
        return preprocess_image(img_u8), mask_u8.astype("float32")
    x   = img_u8 if img_u8.ndim == 3 else img_u8[..., None]
    out = aug(image=x, mask=mask_u8)
    return preprocess_image(out["image"]), out["mask"].astype("float32")

# ── 噪聲函式 ──────────────────────────────────────────────────
def make_gauss_noise(std):
    def _fn(image, **kw):
        return np.clip(image.astype(np.float32) + np.random.normal(0, std, image.shape), 0, 255).astype(np.uint8)
    return A.Lambda(image=_fn, p=1.0)

def make_salt_pepper(p_thr, s_thr):
    def _fn(image, **kw):
        img = image.copy(); r = np.random.rand(*image.shape[:2])
        img[r < p_thr] = 0; img[r > s_thr] = 255
        return img
    return A.Lambda(image=_fn, p=1.0)

def build_train_aug(noise_setting="none"):
    ops = [
        A.HorizontalFlip(p=0.5),
        A.Affine(translate_percent={"x":(-0.05,0.05),"y":(-0.05,0.05)},
                 scale=(0.95,1.05), rotate=(-10,10), p=0.8),
    ]
    if noise_setting == "low":
        ops.append(A.Compose([make_gauss_noise(20), make_salt_pepper(0.05, 0.95)], p=1.0))
    elif noise_setting == "high":
        ops.append(A.Compose([A.GaussianBlur(blur_limit=(5,5), p=1.0),
                               make_gauss_noise(50), make_salt_pepper(0.10, 0.90)], p=1.0))
    return A.Compose(ops)

def build_test_aug(noise_setting="none"):
    if noise_setting in ["none", None]: return A.Compose([])
    if noise_setting == "low":  return A.Compose([make_gauss_noise(20), make_salt_pepper(0.05,0.95)])
    if noise_setting == "high": return A.Compose([A.GaussianBlur(blur_limit=(5,5),p=1.0),
                                                   make_gauss_noise(50), make_salt_pepper(0.10,0.90)])
    raise ValueError(f"Unknown noise_setting: {noise_setting}")


In [ ]:
def gen(ids, aug=None, class_idx=None):
    while True:
        rng = np.random.RandomState()
        order = rng.permutation(len(ids))
        for i in order:
            img_id   = ids[i]
            df_group = groups.get_group(img_id) if img_id in groups.groups else None
            img, mask = load_image_and_mask(img_id, df_group)  # 直接拿整張圖
            img_f, mask_f = apply_aug(img, mask, aug)
            if class_idx is not None:
                mask_f = mask_f[..., class_idx:class_idx+1]
            yield img_f, mask_f

def make_ds(ids, aug, batch=BATCH, shuffle=False, repeat=False, class_idx=None):
    if class_idx is not None:
        sig = (tf.TensorSpec((IMG_H, IMG_W, 3), tf.float32),
               tf.TensorSpec((IMG_H, IMG_W, 1), tf.float32))  # ← 1 channel
    else:
        sig = (tf.TensorSpec((IMG_H, IMG_W, 3), tf.float32),
               tf.TensorSpec((IMG_H, IMG_W, N_CLASSES), tf.float32))

    ds = tf.data.Dataset.from_generator(
        lambda: gen(ids, aug, class_idx=class_idx), output_signature=sig)
    if shuffle: ds = ds.shuffle(1024, seed=RANDOM_SEED, reshuffle_each_iteration=True)
    if repeat:  ds = ds.repeat()
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

train_aug = build_train_aug(NOISE_SETTING)
val_aug   = build_test_aug(NOISE_SETTING)

train_ds = make_ds(tr_ids, train_aug, shuffle=True,  repeat=True)
val_ds   = make_ds(va_ids, val_aug,   shuffle=False, repeat=False)
test_ds  = make_ds(te_ids, val_aug,   shuffle=False, repeat=False)
print(f"Dataset 建立完成 — train={len(tr_ids)}  val={len(va_ids)}  test={len(te_ids)}")


Dataset 建立完成 — train=19743  val=2514  test=2514


## Loss Function

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable(package="custom", name="macro_double_soft_dice_loss")
def macro_double_soft_dice_loss(y_true, y_pred, eps=1e-7):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    inter = tf.reduce_sum(y_true * y_pred, axis=[1, 2])      # (B,C)
    denom = tf.reduce_sum(y_true + y_pred, axis=[1, 2])      # (B,C)
    dice  = (2.0 * inter + eps) / (denom + eps)              # (B,C)

    return 1.0 - tf.reduce_mean(dice)                        # macro over B and C


@register_keras_serializable(package="custom", name="per_class_focal_loss")
def per_class_focal_loss(y_true, y_pred, alpha=(0.6, 0.85, 0.35, 0.35), gamma=2.0, eps=1e-7):
    """
    alpha: (a1,a2,a3,a4) 對應 Defect1~4
    直覺：Defect2 最稀有 → 給最大 alpha
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), eps, 1.0 - eps)

    alpha = tf.constant(alpha, dtype=tf.float32)          # (C,)
    alpha = tf.reshape(alpha, (1, 1, 1, -1))              # broadcast -> (1,1,1,C)

    ce = - (y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))

    pt = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)

    # per-class alpha weighting
    weight = alpha * y_true + (1.0 - alpha) * (1.0 - y_true)

    fl = weight * tf.pow(1.0 - pt, gamma) * ce

    return tf.reduce_mean(fl)


@register_keras_serializable(package="custom", name="combined_loss_pc")
def combined_loss(y_true, y_pred):
    return per_class_focal_loss(y_true, y_pred) + macro_double_soft_dice_loss(y_true, y_pred)

# @register_keras_serializable(package="custom", name="combined_loss")
# def combined_loss(y_true, y_pred, eps=1e-7):
#     y_true = tf.cast(y_true, tf.float32)
#     y_pred = tf.cast(y_pred, tf.float32)
#     inter = tf.reduce_sum(y_true * y_pred, axis=[1,2])
#     denom = tf.reduce_sum(y_true + y_pred, axis=[1,2])
#     return 1.0 - tf.reduce_mean((2.0*inter + eps)/(denom + eps))

@register_keras_serializable(package="custom", name="macro_dice_score")
def macro_dice_score(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # 1. 計算每個類別的 Intersection (只在 H, W 加總)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    denominator = tf.reduce_sum(y_true + y_pred, axis=[1, 2])

    # 2. 計算每個樣本每個類別的 Dice Score
    dice_score = (2.0 * intersection + 1e-7) / (denominator + 1e-7)

    # 3. 先對 Batch 取平均，再對 4 個類別取平均 (Macro Average)
    return tf.reduce_mean(dice_score)

@register_keras_serializable(package="custom", name="macro_iou_score")
def macro_iou_score(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2]) - intersection

    iou_score = (intersection + 1e-7) / (union + 1e-7)

    return tf.reduce_mean(iou_score)

#＃建立模型架構

In [ ]:
# ================================================================
# Attention Gate
# ================================================================
# 修改重點：
#   - attention_gate()：新增模組，用 decoder gating signal 過濾 skip
#   - up_block()：在 concat 前插入 AG，其餘不變
#   - build_unet() / build_model()：完全不動
# ================================================================

import tensorflow as tf
import tensorflow.keras.layers as L
import tensorflow.keras as keras



# ──────────────────────────────────────────────────────────────────
# 1. Backbone（與原本相同，支援三種 backbone）
# ──────────────────────────────────────────────────────────────────
def get_backbone(x, name):
    if name == "resnet34":                         # ← 新增
        bb, feats = build_resnet34_backbone(x)       # 無預訓練權重

    elif name == "inceptionv3":
        bb    = tf.keras.applications.InceptionV3(
                    include_top=False, weights="imagenet", input_tensor=x)
        feats = [bb.get_layer(n).output
                 for n in ["mixed2", "mixed5", "mixed8", "mixed10"]]

    elif name == "efficientnetb4":
        bb    = tf.keras.applications.EfficientNetB4(
                    include_top=False, weights="imagenet", input_tensor=x)
        feats = [bb.get_layer(n).output
                 for n in ["block2b_add", "block3b_add",
                            "block5c_add", "block7b_add"]]
    else:
        raise ValueError(f"Unknown backbone: {name}")

    for i, f in enumerate(feats, 1):
        print(f"  c{i}: {f.shape}")

    return bb, feats


# ──────────────────────────────────────────────────────────────────
# 2. 基本卷積塊（與原本相同）
# ──────────────────────────────────────────────────────────────────
def conv_block(x, filters, k=3):
    x = L.Conv2D(filters, k, padding="same")(x)
    x = L.BatchNormalization()(x)
    return L.Activation("relu")(x)


"""
    Modified Attention Gate (MAG)：基於 Oktay et al. (2018) Attention U-Net
    的 Attention Gate 進行修改，用於本研究 CSAD U-Net 的 skip connection 過濾。

    ── 與 Oktay et al. (2018) 原始設計的差異 ──────────────────────────

    差異 1：新增 3 個 BatchNormalization
        Oktay 原始公式（無 BN）：
            q = ψ^T · ReLU(W_x^T·x + W_g^T·g + b_g) + b_ψ
            α = Sigmoid(q)
        本研究公式（3 處皆加入 BN）：
            α = Sigmoid( BN( W_ψ^T · ReLU( BN(W_x^T·x) + BN(W_g^T·g) ) ) )
        動機：穩定訓練過程穩定 α 分布，
              加入 BN 後可穩定每一層的特徵尺度，加速收斂。

    差異 2：x 與 g 的解析度對齊時機不同
        Oktay 原始版本：x（encoder，高解析度）與 g（decoder，低解析度）
            解析度不同，AG 內部用 strided conv 將 x 降採樣到與 g 相同
            解析度計算注意力係數，再用 trilinear upsample 把 α 放大回 x
            的原始解析度。
        本研究版本：解析度對齊已在呼叫此函式「之前」的 up_block() 中，
            用 L.Resizing() 把 decoder 特徵對齊到 skip 的解析度，因此
            進入本函式時 x 與 g 已經是相同的 (H, W)，本函式不需要處理
            降採樣／升採樣，屬簡化版設計。

    差異 3：具名注意力圖輸出（attn_map_{name}）
        Oktay 原始版本沒有這個設計。
        為了 4-2 節 GradCAM可解釋性分析與注意力視覺化需求，將每個 attention map 依
        「類別-層級」命名（例如 attn_map_c0_up1），方便訓練後
        個別取出檢視、比較四個類別專屬解碼器的注意力聚焦位置。

    Args:
        x: skip feature，來自 encoder（高解析度、細節豐富）
        g: gating signal，來自 decoder（已對齊解析度、語意清晰）
        inter_channels: 中間投影維度，通常 = max(filters // 2, 16)
        name: 用於區分不同類別/層級的注意力圖命名

    Returns:
        過濾後的 skip feature（x̂ = α ⊗ x）
"""
def attention_gate(x, g, inter_channels, name=""):  # ← 加 name 參數
# ── Step 1：x, g 各自投影到中間維度 + BN（差異 1：Oktay 原始無 BN）──
    Wx = L.Conv2D(inter_channels, kernel_size=1, padding="same")(x)
    Wx = L.BatchNormalization()(Wx)
    Wg = L.Conv2D(inter_channels, kernel_size=1, padding="same")(g)
    Wg = L.BatchNormalization()(Wg)
# ── Step 2：融合 x, g 的資訊，ReLU 非線性 ──────────────────────
    psi = L.Add()([Wx, Wg])
    psi = L.Activation("relu")(psi)
# ── Step 3：壓成 1 channel + BN（差異 1：Oktay 原始 ψ 後無 BN）──
    psi = L.Conv2D(1, kernel_size=1, padding="same")(psi)
    psi = L.BatchNormalization()(psi)

# ── Step 4：Sigmoid 得到注意力係數 α，具名輸出（差異 3）────────
    alpha = L.Activation("sigmoid", name=f"attn_map_{name}")(psi)
# ── Step 5：α ⊗ x，過濾後的 skip feature（與 Oktay 原始相同）───
    return L.Multiply()([x, alpha])


# ──────────────────────────────────────────────────────────────────
# 4. Decoder 塊（加入 AG，其餘邏輯與原本相同）
#
#   原本：upsample → concat(skip) → conv → conv
#   現在：upsample → AG(skip, g=upsample後的x) → concat → conv → conv
# ──────────────────────────────────────────────────────────────────
    """
    Decoder 上採樣塊，包含 MAG 過濾後的 skip connection。

    差異 2 的實作位置：Oktay 原始版本在 attention_gate() 內部處理
    x（skip）與 g（decoder）的解析度對齊；本研究改在此函式先用
    Resizing 對齊好，再呼叫 attention_gate()，讓 MAG 本身邏輯更單純。
    """
def up_block(x, skip, filters, name=""):
    x = L.UpSampling2D((2, 2), interpolation="bilinear")(x)
    if skip is not None:
        # ★ 加這行對齊尺寸（InceptionV3 需要，其他 backbone 也不影響）
        skip_shape = skip.shape[1:3]
        x = L.Resizing(skip_shape[0], skip_shape[1],
                       interpolation="bilinear")(x)

        inter_ch = max(filters // 2, 16)
        skip = attention_gate(x=skip, g=x,
                             inter_channels=inter_ch,
                             name=name)
        x = L.Concatenate()([x, skip])
    return conv_block(conv_block(x, filters), filters)

def residual_block(x, filters, stride=1, downsample=False):
    shortcut = x
    x = L.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)
    x = L.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    if downsample:
        shortcut = L.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = L.BatchNormalization()(shortcut)
    x = L.Add()([x, shortcut])
    return L.Activation("relu")(x)


def build_resnet34_backbone(input_tensor):
    """
    ResNet34 骨幹網路，回傳四個 stage 的 feature map。
    對應輸入尺寸 128×800：
        c1: (32, 200, 64)
        c2: (16, 100, 128)
        c3: (8,  50,  256)
        c4: (4,  25,  512)
    注意：無 ImageNet 預訓練權重，從頭訓練。
    """
    # Stem
    x = L.Conv2D(64, 7, strides=2, padding="same", use_bias=False)(input_tensor)
    x = L.BatchNormalization()(x)
    x = L.Activation("relu")(x)
    x = L.MaxPooling2D(3, strides=2, padding="same")(x)

    # Stage 1 → c1（64 filters, 3 blocks, stride 1）
    for _ in range(3):
        x = residual_block(x, 64)
    c1 = x

    # Stage 2 → c2（128 filters, 4 blocks, stride 2）
    x = residual_block(x, 128, stride=2, downsample=True)
    for _ in range(3):
        x = residual_block(x, 128)
    c2 = x

    # Stage 3 → c3（256 filters, 6 blocks, stride 2）
    x = residual_block(x, 256, stride=2, downsample=True)
    for _ in range(5):
        x = residual_block(x, 256)
    c3 = x

    # Stage 4 → c4（512 filters, 3 blocks, stride 2）
    x = residual_block(x, 512, stride=2, downsample=True)
    for _ in range(2):
        x = residual_block(x, 512)
    c4 = x

    # 不建立多餘的 sub-model，用 dummy 物件佔位
    # （build_unet / build_fpn 只使用 feats，不使用 bb）
    class _Backbone:
        pass
    bb = _Backbone()

    return bb, [c1, c2, c3, c4]


# ──────────────────────────────────────────────────────────────────
# 5. 完整 UNet
# ──────────────────────────────────────────────────────────────────

def build_unet(backbone_name):
    inp = L.Input((IMG_H, IMG_W, 3))
    _, (c1, c2, c3, c4) = get_backbone(inp, backbone_name)
    c4 = conv_block(c4, 512)

    # 4 個獨立 decoder，每個輸出 1 channel
    outputs = []
    for c in range(4):
        x = up_block(c4, c3, 256, name=f"c{c}_up1")
        x = up_block(x,  c2, 128, name=f"c{c}_up2")
        x = up_block(x,  c1,  64, name=f"c{c}_up3")
        x = up_block(x, None, 32, name=f"c{c}_up4")
        x = L.Resizing(IMG_H, IMG_W, interpolation="bilinear")(x)
        x = L.Conv2D(1, 1, name=f"head_{c}")(x)
        x = L.Activation("sigmoid", name=f"out_{c}")(x)
        outputs.append(x)

    return keras.Model(inp, outputs, name=f"unet_ag_{backbone_name}")

# # ──────────────────────────────────────────────────────────────────
# # 6. 編譯（與原本相同）
# # ──────────────────────────────────────────────────────────────────
def build_model():
    if ARCH == "unet":
        m = build_unet(BACKBONE)
        m.compile(
            optimizer=keras.optimizers.Adam(LR),
            loss=[combined_loss] * 4,
            loss_weights=[1.0, 1.0, 1.0, 1.0],
            metrics=[[macro_dice_score, macro_iou_score], [], [], []],
        )
    return m

model = build_model()
model.summary(line_length=120)
print("backbone:", BACKBONE)
print("model:    ", f"{ARCH}_ag_shared")
print("noise:   ", NOISE_SETTING)


  c1: (None, 32, 200, 32)
  c2: (None, 16, 100, 56)
  c3: (None, 8, 50, 160)
  c4: (None, 4, 25, 448)


Model: "unet_ag_efficientnetb4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                      ┃ Output Shape                 ┃           Param # ┃ Connected to              
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ input_layer_4 (InputLayer)        │ (None, 128, 800, 3)          │                 0 │ -                         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ rescaling_8 (Rescaling)           │ (None, 128, 800, 3)          │                 0 │ input_layer_4[0][0]       
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ normalization_4 (Normalization)   │ (None, 128, 800, 3)          │                 7 │ rescaling_8[0][0]         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ rescaling_9 (Rescaling)           │ (None, 128, 800, 3)          │                 0 │ normalization_4[0][0]     
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ stem_conv_pad (ZeroPadding2D)     │ (None, 129, 801, 3)          │                 0 │ rescaling_9[0][0]         
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ stem_conv (Conv2D)                │ (None, 64, 400, 48)          │             1,296 │ stem_conv_pad[0][0]       
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ stem_bn (BatchNormalization)      │ (None, 64, 400, 48)          │               192 │ stem_conv[0][0]           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ stem_activation (Activation)      │ (None, 64, 400, 48)          │                 0 │ stem_bn[0][0]             
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_dwconv (DepthwiseConv2D)  │ (None, 64, 400, 48)          │               432 │ stem_activation[0][0]     
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_bn (BatchNormalization)   │ (None, 64, 400, 48)          │               192 │ block1a_dwconv[0][0]      
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_activation (Activation)   │ (None, 64, 400, 48)          │                 0 │ block1a_bn[0][0]          
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_se_squeeze                │ (None, 48)                   │                 0 │ block1a_activation[0][0]  
│ (GlobalAveragePooling2D)          │                              │                   │                           
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_se_reshape (Reshape)      │ (None, 1, 1, 48)             │                 0 │ block1a_se_squeeze[0][0]  
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_se_reduce (Conv2D)        │ (None, 1, 1, 12)             │               588 │ block1a_se_reshape[0][0]  
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_se_expand (Conv2D)        │ (None, 1, 1, 48)             │               624 │ block1a_se_reduce[0][0]   
├───────────────────────────────────┼──────────────────────────────┼───────────────────┼───────────────────────────
│ block1a_se_excite (Multiply)      │ (None, 64, 400, 48

 Total params: 30,611,103 (116.77 MB)

 Trainable params: 30,477,168 (116.26 MB)

 Non-trainable params: 133,935 (523.19 KB)

backbone: efficientnetb4
model:     unet_ag_shared
noise:    none


In [ ]:
# ============================================================
# 訓練資料管線：將 4-channel mask 拆成對應四個獨立解碼器輸出頭
# ============================================================

N_CLASSES = 4

steps_per_epoch = max(1, math.ceil(len(tr_ids) / BATCH))
validation_steps = max(1, math.ceil(len(va_ids) / BATCH))


def wrap_ds(ds):
    """將 (image, 4-channel mask) 轉成 (image, {out_0: mask0, ..., out_3: mask3})，
    對應 CSAD U-Net 四個類別專屬解碼器的輸出格式。"""
    return ds.map(lambda x, y: (
        x, {
            "out_0": y[..., 0:1],
            "out_1": y[..., 1:2],
            "out_2": y[..., 2:3],
            "out_3": y[..., 3:4],
        }
    ))


train_ds_w = wrap_ds(make_ds(tr_ids, train_aug, shuffle=True, repeat=True))
val_ds_w = wrap_ds(make_ds(va_ids, val_aug, shuffle=False, repeat=False))


# ============================================================
# 訓練
# ============================================================

ckpt_path = f"{MODEL_DIR}/unet_ag_shared_{BACKBONE}.keras"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        ckpt_path,
        monitor="val_out_0_macro_iou_score",  # 以 Defect 1 驗證 IoU 作為 checkpoint 篩選指標
        mode="max", save_best_only=True, verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_out_0_macro_iou_score",
        factor=0.5, patience=5, mode="max", verbose=1,
    ),
]

model.fit(
    train_ds_w,
    validation_data=val_ds_w,
    epochs=EPOCHS,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
)

print(f"✅ 模型已儲存：{ckpt_path}")

Epoch 1/30


KeyboardInterrupt: 